In [7]:

import pandas as pd
import spacy
from spellchecker import SpellChecker
import re
import importlib
import utils
import calculate_linguisticFeatures
import numpy as np

importlib.reload(utils)
importlib.reload(calculate_linguisticFeatures)

<module 'calculate_linguisticFeatures' from 'c:\\Projetos\\mineracao_arquivos_epstein_2026\\calculate_linguisticFeatures.py'>

In [12]:
# dataset10 = utils.Utils.load_from_dir()
# dataset10.to_csv(getcwd()+'/datasets/dataset10.csv', sep='|', index=False)

In [182]:

def count_misspellings(doc):
    """Identifica palavras com erros ortográficos em um texto"""
    
    spell = SpellChecker(language='en')  # Definir o idioma como portuguêsn
    
    # Conta o número de erros ortográficos para cada texto na lista
    misspellings_counts = len(spell.unknown([token.text for token in doc if token.is_alpha]))
    
    return misspellings_counts

def check_and_load_spacy_model(model_name):
    """Verifica se o modelo está instalado, caso contrário, instala automaticamente.
    """
    
    # Verifica se o modelo está disponível
    if model_name not in spacy.util.get_installed_models():
        print(f"Modelo {model_name} não encontrado. Instalando...")
        spacy.cli.download(model_name)
    
    # Carrega o modelo
    nlp = spacy.load(model_name)
    
    return nlp

In [ ]:
dt_raw = utils.Utils.load()

In [184]:
# dt1 = pd.read_csv('datasets1.csv', sep='|', index_col=0)
# dt2 = pd.read_csv('datasets2.csv', sep='|', index_col=0)
# pd.concat([dt1, dt2])[['dataset', 'file', 'content', 'file_type', 'len', 'count_tokens',
#        'count_misspellings', 'count_sentences']].to_csv('datasets-misspellings.csv', sep='|')

In [185]:
dt_raw['file_type'] = dt_raw['file'].apply(lambda x: x.split('.')[-1])
dt_raw['file_type'] = dt_raw['file_type'].replace({
    'mp4':'video',
    'avi':'video',
    'mp3':'audio',
    'm4a':'audio',
    'xlsx': 'sheet',
    'xls': 'sheet',
    'csv': 'sheet',
})
#PDFs sem texto são todos imagens
dt_raw.loc[dt_raw['content'].isna(), 'file_type'] = 'image'

#dropar duplicatas vazias que não são imagens
dt_raw.loc[dt_raw['file_type'] == 'pdf', 'content'] = dt_raw.loc[dt_raw['file_type'] == 'pdf', 'content'].replace({'failed': np.nan})
dt_raw = dt_raw.sort_values('content', na_position='first')
dt_raw = dt_raw.drop_duplicates(keep='last')

dt_raw['file_type'].value_counts().to_frame().style

,count
file_type,
pdf,12372
image,2388
video,420
sheet,16
audio,4


In [191]:
dt = dt_raw.loc[dt_raw['file_type'] == 'pdf']
dt = dt.dropna()

In [192]:
linguistics = calculate_linguisticFeatures.linguisticFeatures()

In [193]:
linguistics.nlpModel.max_length = 1257000

In [194]:
dt['len'] = dt['content'].apply(lambda x: len(x))
dt = dt.sort_values('len', ascending=False)
# dt = dt.loc[dt['len'] < 100000]

In [ ]:
dt.loc[0, 'preprocessed_text'] = ''
for i in range(dt.shape[0]):
  dtemp = linguistics.get(
      dt['content'].iloc[i],
      'preprocess',
  )
  dt.iloc[i, 5] = dtemp['preprocess'][0]
  if i%100 == 0:
    dt.to_csv(f'datasets-preprocess.csv', sep='|', index=False)
dt.to_csv(f'datasets-preprocess.csv', sep='|', index=False)

In [212]:
dt

,dataset,file,content,file_type,len,preprocessed_text
1972,8,EFTA00017143.pdf,MOBILITY 411Olo\nAT&T\n2996479\n06/17/2020 AT&...,pdf,1256602,MOBILITY 411Olo AT&T 2996479 06/17/2020 AT&T q...
496,8,EFTA00012111.pdf,DocuSign Envelope ID: 5F5A5466-1857-4351a244-A...,pdf,1048487,DocuSign Envelope 5f5a5466 1857 4351a244 a6fd4...
48,12,EFTA02857863.pdf,ED-1057 (Rev. 5-8-10)\n(Overall Document Class...,pdf,686671,ED-1057 Rev. Overall Document Classification R...
495,8,EFTA00011669.pdf,OM No.21200020 Ettoternie TM:alp Wetter\nEEmxp...,pdf,501326,No.21200020 Ettoternie alp Wetter EEmxps 541/2...
142,12,EFTA02857524.pdf,Memorandum\nSubject Date\nRe: Operation Leap Y...,pdf,382620,Memorandum Subject Date Operation Leap Year 20...
...,...,...,...,...,...,...
78,2,EFTA00003333.pdf,1\n,pdf,2,
307,2,EFTA00003581.pdf,I\n,pdf,2,
557,2,EFTA00003849.pdf,I\n,pdf,2,
172,2,EFTA00003429.pdf,a\n,pdf,2,


In [213]:
dt.loc[0, ['count_tokens', 'count_misspellings', 'count_sentences']] = 0
for i in range(dt.shape[0]):
  dtemp = linguistics.get(
      dt['preprocessed_text'].iloc[i],
      'count_tokens',
      'count_misspellings',
      'count_sentences',
  )
  dt.iloc[i, [6,7,8]] = dtemp
  if i%100 == 0:
    dt.to_csv(f'datasets-misspellings.csv', sep='|', index=False)
dt.to_csv(f'datasets-misspellings.csv', sep='|', index=False)

In [214]:
dt.to_csv(f'datasets-misspellings.csv', sep='|', index=False)

In [22]:
dt

,Unnamed: 0,dataset,file,content,file_type,len,count_tokens,count_misspellings,count_sentences
0,1972,8,EFTA00017143.pdf,MOBILITY 411Olo\nAT&T\n2996479\n06/17/2020 AT&...,pdf,1256602,250790.0,388.0,11817.0
1,496,8,EFTA00012111.pdf,DocuSign Envelope ID: 5F5A5466-1857-4351a244-A...,pdf,1048487,232853.0,2444.0,6597.0
2,495,8,EFTA00011669.pdf,OM No.21200020 Ettoternie TM:alp Wetter\nEEmxp...,pdf,501326,113905.0,4083.0,5571.0
3,4720,8,EFTA00025624.pdf,Return Information\nWARNING\nSchedule E. Unrec...,pdf,334333,76838.0,652.0,4154.0
4,4164,8,EFTA00024032.pdf,IN THE UNITED STATES DISTRICT COURT\nFOR THE S...,pdf,328115,64772.0,136.0,1572.0
...,...,...,...,...,...,...,...,...,...
12367,114,12,EFTA02731704.pdf,School Choice Week\nD> Navigate Schools with E...,pdf,49,NaN,NaN,NaN
12368,128,12,EFTA02731726.pdf,From:\nTo:\nSubject Accepted: Leon Mac ca\n,pdf,40,NaN,NaN,NaN
12369,149,12,EFTA02731780.pdf,WIGDOR\n,pdf,7,NaN,NaN,NaN
12370,56,12,EFTA02731500.pdf,WIGDOR\n,pdf,7,NaN,NaN,NaN


In [29]:
print('arquivos sem texto: ', dt['content'].isna().sum())
print((dt['count_tokens'].isna().sum()/dt['content'].size).round(2), '%')

arquivos sem texto:  0
0.01 %


In [35]:
dt['ratio_misspelling'] = dt['count_misspellings']/dt['count_tokens']
dt.sort_values('ratio_misspelling', ascending=False)

,Unnamed: 0,dataset,file,content,file_type,len,count_tokens,count_misspellings,count_sentences,ratio_misspelling
88,3060,8,EFTA00020596.pdf,Dateimmddyyyyj Box Number Form Type = 'CITADEL...,pdf,52626,12293.0,1693.0,597.0,0.137721
181,5832,8,EFTA00029206.pdf,stArr or onArrAAS\nssenTARY or tiAn\naK u 135F...,pdf,32953,8321.0,1141.0,558.0,0.137123
287,3276,8,EFTA00021076.pdf,q\nIto cf\nNUSSnc~c Peke:lc\nSA0 300 g ii/Jr6\...,pdf,21461,6541.0,619.0,354.0,0.094634
47,3472,8,EFTA00021578.pdf,NOW NeitlerNartn\n560 818 Alyea 1 Na 'NAV 102S...,pdf,90628,31526.0,2148.0,1743.0,0.068134
118,73,4,EFTA00007824.pdf,"II ' III 1,1111 1\n/0Q5--\nz\n1\n18E 10 0018 1...",pdf,43284,10635.0,685.0,598.0,0.064410
...,...,...,...,...,...,...,...,...,...,...
12367,114,12,EFTA02731704.pdf,School Choice Week\nD> Navigate Schools with E...,pdf,49,NaN,NaN,NaN,NaN
12368,128,12,EFTA02731726.pdf,From:\nTo:\nSubject Accepted: Leon Mac ca\n,pdf,40,NaN,NaN,NaN,NaN
12369,149,12,EFTA02731780.pdf,WIGDOR\n,pdf,7,NaN,NaN,NaN,NaN
12370,56,12,EFTA02731500.pdf,WIGDOR\n,pdf,7,NaN,NaN,NaN,NaN


In [32]:
dt['count_tokens'].isna().sum()

np.int64(150)